# Feature Engineering
Encoding & Dataset Preparation



In [15]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv('C:/Users/Stepheny/Desktop/Projects/Customer_Churn/data/cleaned_df.csv')

print(df.shape)
print(df.head())
print(df.columns)
df.info()


(7043, 19)
   gender  SeniorCitizen Partner Dependents  tenure     MultipleLines  \
0  Female              0     Yes         No       1  No phone service   
1    Male              0      No         No      34                No   
2    Male              0      No         No       2                No   
3    Male              0      No         No      45  No phone service   
4  Female              0      No         No       2                No   

  InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport  \
0             DSL             No          Yes               No          No   
1             DSL            Yes           No              Yes          No   
2             DSL            Yes          Yes               No          No   
3             DSL            Yes           No              Yes         Yes   
4     Fiber optic             No           No               No          No   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No  

In [3]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].unique())



gender:
['Female' 'Male']

Partner:
['Yes' 'No']

Dependents:
['No' 'Yes']

MultipleLines:
['No phone service' 'No' 'Yes']

InternetService:
['DSL' 'Fiber optic' 'No']

OnlineSecurity:
['No' 'Yes' 'No internet service']

OnlineBackup:
['Yes' 'No' 'No internet service']

DeviceProtection:
['No' 'Yes' 'No internet service']

TechSupport:
['No' 'Yes' 'No internet service']

StreamingTV:
['No' 'Yes' 'No internet service']

StreamingMovies:
['No' 'Yes' 'No internet service']

Contract:
['Month-to-month' 'One year' 'Two year']

PaperlessBilling:
['Yes' 'No']

PaymentMethod:
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Churn:
['No' 'Yes']


| Column             | Encoding        |
| ------------------ | --------------- |
| `customerID`       | Drop            |
| `gender`           | Label           |
| `SeniorCitizen`    | Keep numeric    |
| `Partner`          | Label           |
| `Dependents`       | Label           |
| `tenure`           | Numeric         |
| `PhoneService`     | Drop            |
| `MultipleLines`    | One-Hot         |
| `InternetService`  | One-Hot         |
| `OnlineSecurity`   | One-Hot         |
| `OnlineBackup`     | One-Hot         |
| `DeviceProtection` | One-Hot         |
| `TechSupport`      | One-Hot         |
| `StreamingTV`      | One-Hot         |
| `StreamingMovies`  | One-Hot         |
| `Contract`         | Ordinal         |
| `PaperlessBilling` | Label           |
| `PaymentMethod`    | One-Hot         |
| `MonthlyCharges`   | Numeric         |
| `TotalCharges`     | Numeric         |
| `Churn`            | Label           |


In [4]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Grouping features

In [5]:
binary_cols = ["gender", "Partner", "Dependents", "PaperlessBilling"]

ordinal_cols = ["Contract"]
contract_order = [["Month-to-month", "One year", "Two year"]]

nominal_cols = [
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "PaymentMethod"
]

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]


Preprocessing

In [6]:


preprocessor = ColumnTransformer(
    transformers=[
        ("bin", OneHotEncoder(drop="if_binary"), binary_cols),
        ("ord", OrdinalEncoder(categories=contract_order), ordinal_cols),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)


Logistic Regression

In [7]:


logit = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000
    ))
])

logit.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('bin',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['gender', 'Partner',
                                                   'Dependents',
                                                   'PaperlessBilling']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['Month-to-month',
                                                                              'One '
                                                                              'year',
                                                                              'Two '
                                                                              'year']]),
                                                  ['Contract']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'PaymentMethod']),
                                                 ('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'SeniorCitizen'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

Random Forest

In [8]:


rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=15,
        min_samples_leaf=10,
        max_features=0.5,
        class_weight="balanced",
        random_state=42
    ))
])

rf.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('bin',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['gender', 'Partner',
                                                   'Dependents',
                                                   'PaperlessBilling']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['Month-to-month',
                                                                              'One '
                                                                              'year',
                                                                              'Two '
                                                                              'year']]),
                                                  ['Contract']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'PaymentMethod']),
                                                 ('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'SeniorCitizen'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=15,
                                        max_features=0.5, min_samples_leaf=10,
                                        n_estimators=400, random_state=42))])

Gradient Boosting XGBoost 

In [9]:


scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()

xgb = Pipeline([
    ("prep", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('bin',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['gender', 'Partner',
                                                   'Dependents',
                                                   'PaperlessBilling']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['Month-to-month',
                                                                              'One '
                                                                              'year',
                                                                              'Two '
                                                                              'year']]),
                                                  ['Contract']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBac...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=400, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

Evaluation

In [10]:


models = {
    "Logistic": logit,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name, model in models.items():
    probs = model.predict_proba(X_test)[:,1]
    print(f"{name}: ROC-AUC = {roc_auc_score(y_test, probs):.3f} | PR-AUC = {average_precision_score(y_test, probs):.3f}")


Logistic: ROC-AUC = 0.842 | PR-AUC = 0.633
Random Forest: ROC-AUC = 0.841 | PR-AUC = 0.648
XGBoost: ROC-AUC = 0.832 | PR-AUC = 0.644


Probability Calibration 

In [11]:


calibrated_xgb = CalibratedClassifierCV(xgb, method="isotonic", cv=5)
calibrated_xgb.fit(X_train, y_train)


CalibratedClassifierCV(cv=5,
                       estimator=Pipeline(steps=[('prep',
                                                  ColumnTransformer(transformers=[('bin',
                                                                                   OneHotEncoder(drop='if_binary'),
                                                                                   ['gender',
                                                                                    'Partner',
                                                                                    'Dependents',
                                                                                    'PaperlessBilling']),
                                                                                  ('ord',
                                                                                   OrdinalEncoder(categories=[['Month-to-month',
                                                                                                               'One '
                                                                                                               'year',
                                                                                                               'Two '
                                                                                                               'year']]),
                                                                                   ['Contract']),
                                                                                  ('nom',
                                                                                   OneHotEncoder(handle_unknown='ignore'),
                                                                                   ['MultipleLines',
                                                                                    'Intern...
                                                                importance_type=None,
                                                                interaction_constraints=None,
                                                                learning_rate=0.05,
                                                                max_bin=None,
                                                                max_cat_threshold=None,
                                                                max_cat_to_onehot=None,
                                                                max_delta_step=None,
                                                                max_depth=5,
                                                                max_leaves=None,
                                                                min_child_weight=None,
                                                                missing=nan,
                                                                monotone_constraints=None,
                                                                multi_strategy=None,
                                                                n_estimators=400,
                                                                n_jobs=None,
                                                                num_parallel_tree=None,
                                                                random_state=42, ...))]),
                       method='isotonic')

Model Selection

In [12]:
final_model = calibrated_xgb


In [14]:
joblib.dump(final_model, "churn_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")


['preprocessor.pkl']